# 06 -- Final Figures and Tables

**No new computation, no training, no re-derivation of any statistic.**
Every number and figure here comes from files already written to `results/`
by notebooks 00-05 and the CpG unweighted-check notebooks
(`04b_cpg_unweighted_check.ipynb`, `04c_cpg_hotspot_unweighted_check.ipynb`).
If any required input were missing, this notebook would report it explicitly
rather than computing a substitute -- see the input-verification cell below.

**One gap found and fixed before this notebook could be written:**
`03_position_analysis.ipynb` computed a per-position table (`shannon_entropy`,
`accuracy`, `majority_subtype_freq`, `is_cpg`) entirely in-memory and only
ever saved the aggregate `spearman.json` and three rendered PNGs -- the
per-position values themselves were never persisted. Per the user's
decision, notebook 03 was patched with one additional cell that saves
`results/position_analysis/per_position_table.csv`, and that cell was
re-run (not the whole modeling pipeline -- nothing upstream of that point
changed). This notebook reads that file like any other saved input; it does
not recompute it.

**Style conventions used throughout:**
- `hotspot` = blue (`#377eb8`), `rare` = orange (`#ff7f00`) everywhere a
  figure distinguishes the two datasets -- consistent across every figure.
- Reference/baseline quantities (random, majority, ceiling, MCC=0 line) use
  a neutral gray (`#4d4d4d`).
- Font sizes sized for print-journal column width, not slide decks.
- No titles baked into any image -- axis labels and legends only, so the
  paper's figure captions carry the title text instead.
- Every figure saved as both `.png` (300dpi) and `.svg` (vector) to
  `results/figures/`.


In [1]:
SEED = 42
import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
import os
import sys
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data import CLASSES  # fixed label-order constant only, not a computation

RESULTS_ROOT = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(RESULTS_ROOT, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- style conventions, used everywhere ----
COLOR_HOTSPOT = '#377eb8'
COLOR_RARE = '#ff7f00'
COLOR_REFERENCE = '#4d4d4d'
DATASET_COLORS = {'hotspot': COLOR_HOTSPOT, 'rare': COLOR_RARE}

plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

produced_files = []


def save_fig(fig, name):
    png_path = os.path.join(FIGURES_DIR, f'{name}.png')
    svg_path = os.path.join(FIGURES_DIR, f'{name}.svg')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(svg_path, bbox_inches='tight')
    plt.close(fig)
    produced_files.append(png_path)
    produced_files.append(svg_path)
    print(f"Saved -> {png_path}")
    print(f"Saved -> {svg_path}")


print(f"Project root: {PROJECT_ROOT}")
print(f"Figures dir:   {FIGURES_DIR}")


Project root: C:\Users\danya\Documents\projects\tp53_mutation_subtype
Figures dir:   C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures


## Confirm every required input exists before using it

Reports anything missing explicitly rather than silently skipping it or
substituting a computed value.


In [3]:
required_inputs = {
    'main/hotspot/metrics.json': 'results/main/hotspot/metrics.json',
    'main/rare/metrics.json': 'results/main/rare/metrics.json',
    'main/hotspot/confusion_matrix.png': 'results/main/hotspot/confusion_matrix.png',
    'main/rare/confusion_matrix.png': 'results/main/rare/confusion_matrix.png',
    'main/hotspot/predictions.parquet': 'results/main/hotspot/predictions.parquet',
    'main/rare/predictions.parquet': 'results/main/rare/predictions.parquet',
    'main/summary.csv': 'results/main/summary.csv',
    'window_ablation/summary.csv': 'results/window_ablation/summary.csv',
    'position_analysis/spearman.json': 'results/position_analysis/spearman.json',
    'position_analysis/per_position_table.csv': 'results/position_analysis/per_position_table.csv',
    'cpg/metrics.csv': 'results/cpg/metrics.csv',
    'cpg/significance.json': 'results/cpg/significance.json',
    'unweighted/hotspot/metrics.json': 'results/unweighted/hotspot/metrics.json',
    'unweighted/rare/metrics.json': 'results/unweighted/rare/metrics.json',
    'unweighted/summary.csv': 'results/unweighted/summary.csv',
    'cpg/unweighted_check_comparison.csv (rare-CpG)': 'results/cpg/unweighted_check_comparison.csv',
    'cpg/hotspot_unweighted_check_comparison.csv (hotspot-CpG)': 'results/cpg/hotspot_unweighted_check_comparison.csv',
    'repeated_splits/per_split_results.csv': 'results/repeated_splits/per_split_results.csv',
    'repeated_splits/aggregate_summary.csv': 'results/repeated_splits/aggregate_summary.csv',
}

missing = []
for label, rel_path in required_inputs.items():
    full_path = os.path.join(RESULTS_ROOT, *rel_path.split('/')[1:]) if rel_path.startswith('results/') else os.path.join(PROJECT_ROOT, rel_path)
    full_path = os.path.join(PROJECT_ROOT, rel_path)
    exists = os.path.exists(full_path)
    print(f"{'OK ' if exists else 'MISSING'}  {label:<55} -> {full_path}")
    if not exists:
        missing.append(full_path)

print()
if missing:
    raise FileNotFoundError(
        f"{len(missing)} required input(s) are missing -- stopping rather than "
        f"computing a substitute: {missing}"
    )
print("All required inputs present.")


OK   main/hotspot/metrics.json                               -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results/main/hotspot/metrics.json
OK   main/rare/metrics.json                                  -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results/main/rare/metrics.json
OK   main/hotspot/confusion_matrix.png                       -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results/main/hotspot/confusion_matrix.png
OK   main/rare/confusion_matrix.png                          -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results/main/rare/confusion_matrix.png
OK   main/hotspot/predictions.parquet                        -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results/main/hotspot/predictions.parquet
OK   main/rare/predictions.parquet                           -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results/main/rare/predictions.parquet
OK   main/summary.csv                                  

## Class distribution (paper's Figure 1 -- dataset overview)

Source: `data/processed/tp53_mutation_dataset_w21.csv` (the corrected,
906-locus, 693,428-instance combined dataset from notebook 00) -- a direct
`value_counts(normalize=True)` tally, the same kind of simple aggregation
already done for the position-analysis histograms above; no statistic beyond
a frequency count is derived here. This replaces the pre-correction version
of this figure, which was built on the inflated (isoform-fragmented)
position counts and can't be reused as-is.


In [4]:
class_dist_source = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'tp53_mutation_dataset_w21.csv'))
class_pct = (class_dist_source['MutationType'].value_counts(normalize=True) * 100).sort_values(ascending=False)
print(f"n = {len(class_dist_source):,} instances")
print(class_pct.round(2))

fig, ax = plt.subplots(figsize=(4.2, 3.0))
ax.bar(class_pct.index, class_pct.values, color=COLOR_REFERENCE, width=0.6)
for i, v in enumerate(class_pct.values):
    ax.text(i, v + 1.0, f'{v:.1f}%', ha='center', va='bottom', fontsize=6.5)
ax.set_ylabel('Percent of instances')
ax.set_ylim(0, max(class_pct.values) * 1.18)

save_fig(fig, 'fig_class_distribution')


n = 693,428 instances
MutationType
C>T    52.15
C>A    17.14
T>C    12.49
C>G     7.74
T>A     6.00
T>G     4.48
Name: proportion, dtype: float64
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_class_distribution.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_class_distribution.svg


## Figure 1: theoretical ceiling (random / majority / CNN / ceiling)

Source: `results/main/summary.csv` (weighted main-experiment numbers) only
-- no computation. Reference bars (random, majority, ceiling) in gray; the
CNN accuracy bar highlighted in the dataset's own color, with the CNN-to-
ceiling gap annotated directly, since that gap is the figure's point.


In [5]:
main_summary = pd.read_csv(os.path.join(RESULTS_ROOT, 'main', 'summary.csv')).set_index('dataset')

fig, ax = plt.subplots(figsize=(4.2, 3.0))
categories = ['Random', 'Majority', 'CNN', 'Ceiling']
gray_shades = ['#d9d9d9', '#a6a6a6', None, '#4d4d4d']  # CNN slot filled per-dataset below
bar_width = 0.18
group_gap = 0.55
datasets = ['hotspot', 'rare']
x_group_centers = np.arange(len(datasets)) * group_gap * 6

for gi, name in enumerate(datasets):
    row = main_summary.loc[name]
    values = [row['random_accuracy'], row['majority_accuracy'], row['accuracy'], row['position_majority_ceiling']]
    colors = [gray_shades[0], gray_shades[1], DATASET_COLORS[name], gray_shades[3]]
    xs = x_group_centers[gi] + (np.arange(4) - 1.5) * bar_width
    bars = ax.bar(xs, values, width=bar_width, color=colors, edgecolor='black', linewidth=0.4)

    # Annotate the CNN-to-ceiling gap explicitly.
    cnn_x, ceiling_x = xs[2], xs[3]
    cnn_y, ceiling_y = values[2], values[3]
    ax.annotate(
        '', xy=(cnn_x, ceiling_y), xytext=(cnn_x, cnn_y),
        arrowprops=dict(arrowstyle='-|>', color='black', lw=0.8),
    )
    ax.text(cnn_x + 0.02, (cnn_y + ceiling_y) / 2, f"gap\n{ceiling_y - cnn_y:.2f}",
            fontsize=6.5, va='center')

ax.set_xticks(x_group_centers)
ax.set_xticklabels(['hotspot', 'rare'])
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.0)

# Legend built manually so it reflects the semantic categories, not per-dataset color.
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=gray_shades[0], edgecolor='black', linewidth=0.4, label='Random'),
    Patch(facecolor=gray_shades[1], edgecolor='black', linewidth=0.4, label='Majority'),
    Patch(facecolor=COLOR_HOTSPOT, edgecolor='black', linewidth=0.4, label='CNN (hotspot)'),
    Patch(facecolor=COLOR_RARE, edgecolor='black', linewidth=0.4, label='CNN (rare)'),
    Patch(facecolor=gray_shades[3], edgecolor='black', linewidth=0.4, label='Ceiling'),
]
# Legend placed above the axes (outside the plot area) rather than inside a
# corner, since every corner of this plot is occupied by a tall bar (majority
# or ceiling) for at least one dataset.
ax.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=3, frameon=False, columnspacing=1.0, handletextpad=0.4)

save_fig(fig, 'fig_theoretical_ceiling')


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_theoretical_ceiling.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_theoretical_ceiling.svg


## Figure 2: window-size ablation

Source: `results/window_ablation/summary.csv` only. Linear (not log)
x-axis, since only 4 points and the gaps between 11/21/51/101 aren't
multiplicatively even -- plain numeric spacing with explicit tick labels at
each tested window size reads more clearly here than a log axis would.

**Caption note (not baked into the figure -- state this in the paper's
figure caption, not on the plot itself):** the apparent 11bp advantage is
very likely a train/test sequence-similarity artifact (see
`02_window_ablation.ipynb`'s Hamming-distance analysis: test sequences sit
proportionally far closer to training sequences at 11bp than at any larger
window), not evidence of a genuine wider-context signal disappearing at
larger windows. The figure intentionally does not visually emphasize 11bp
as a "win" (no highlighting, no annotation, same marker/line styling as
every other point) -- that qualification belongs in the caption text.


In [6]:
window_summary = pd.read_csv(os.path.join(RESULTS_ROOT, 'window_ablation', 'summary.csv'))

# Shaded band uses the locus-(cluster-)resampled CI, not the instance-resampled
# one: many instances share a locus (identical/near-identical sequence context),
# so instance-level resampling understates uncertainty -- see src/metrics.py's
# bootstrap_accuracy_ci_clustered docstring and Methods -> Evaluation Metrics.
fig, ax = plt.subplots(figsize=(4.2, 3.0))
for name in ['hotspot', 'rare']:
    sub = window_summary[window_summary['dataset'] == name].sort_values('window_size')
    ax.plot(sub['window_size'], sub['accuracy'], marker='o', markersize=4, linewidth=1.2,
            color=DATASET_COLORS[name], label=name)
    ax.fill_between(sub['window_size'], sub['accuracy_ci_low_clustered'], sub['accuracy_ci_high_clustered'],
                     color=DATASET_COLORS[name], alpha=0.15, linewidth=0)

ax.set_xticks(sorted(window_summary['window_size'].unique()))
ax.set_xlabel('Window size (bp)')
ax.set_ylabel('Accuracy')
ax.legend(frameon=False)

save_fig(fig, 'fig_window_ablation')

print("\nCaption note to include in the paper (not rendered on the figure):")
print("  \"The apparent advantage at 11bp is attributable to reduced train/test\"")
print("  \"sequence dissimilarity at short windows rather than genuine wider-context\"")
print("  \"signal (see window-size ablation notebook for the Hamming-distance analysis).\"")


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_window_ablation.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_window_ablation.svg

Caption note to include in the paper (not rendered on the figure):
  "The apparent advantage at 11bp is attributable to reduced train/test"
  "sequence dissimilarity at short windows rather than genuine wider-context"
  "signal (see window-size ablation notebook for the Hamming-distance analysis)."


## Figure 3: position-level analysis (entropy vs. accuracy, entropy distribution, majority-frequency distribution)

Source: `results/position_analysis/per_position_table.csv` and
`spearman.json` only -- both already-saved outputs, no aggregation
performed here. One 3-panel composite figure.


In [7]:
per_position = pd.read_csv(os.path.join(RESULTS_ROOT, 'position_analysis', 'per_position_table.csv'))
spearman = json.load(open(os.path.join(RESULTS_ROOT, 'position_analysis', 'spearman.json')))

fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.0))

# (a) entropy vs accuracy scatter, CpG shown via marker fill
ax = axes[0]
for name in ['hotspot', 'rare']:
    df = per_position[per_position['dataset'] == name]
    non_cpg = df[~df['is_cpg']]
    cpg = df[df['is_cpg']]
    ax.scatter(non_cpg['shannon_entropy'], non_cpg['accuracy'], s=10, alpha=0.45,
               color=DATASET_COLORS[name], linewidths=0)
    ax.scatter(cpg['shannon_entropy'], cpg['accuracy'], s=18, facecolors='none',
               edgecolors=DATASET_COLORS[name], linewidths=0.8)
ax.set_xlabel('Shannon entropy')
ax.set_ylabel('Per-position accuracy')

# Explicit full-opacity legend handles (the real markers are semi-transparent,
# which made the legend swatches nearly invisible / easy to confuse with real
# data points sitting right underneath the legend box).
from matplotlib.lines import Line2D as _Line2D
legend_handles = [
    _Line2D([0], [0], marker='o', color='none', markerfacecolor=DATASET_COLORS[name],
            markeredgecolor='none', markersize=6, label=name, linestyle='none')
    for name in ['hotspot', 'rare']
]
ax.legend(handles=legend_handles, frameon=True, framealpha=0.9, edgecolor='none',
          facecolor='white', loc='best')

# (b) entropy distribution
ax = axes[1]
max_entropy = per_position['shannon_entropy'].max()
bins = np.linspace(0, max_entropy, 25)
for name in ['hotspot', 'rare']:
    ax.hist(per_position.loc[per_position['dataset'] == name, 'shannon_entropy'],
            bins=bins, alpha=0.5, density=True, color=DATASET_COLORS[name], label=name)
ax.set_xlabel('Shannon entropy')
ax.set_ylabel('Density')

# (c) majority-frequency distribution
ax = axes[2]
bins = np.linspace(0, 1, 25)
for name in ['hotspot', 'rare']:
    ax.hist(per_position.loc[per_position['dataset'] == name, 'majority_subtype_freq'],
            bins=bins, alpha=0.5, density=True, color=DATASET_COLORS[name], label=name)
ax.set_xlabel('Majority-subtype frequency')
ax.set_ylabel('Density')

fig.tight_layout()
save_fig(fig, 'fig_position_analysis')

print(f"\nSpearman (for reference, already computed in notebook 03, not recomputed here):")
for name, res in spearman.items():
    print(f"  {name}: rho={res['correlation']:+.4f}  p={res['p_value']:.4g}  n={res['n_positions']:,}")


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_position_analysis.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_position_analysis.svg

Spearman (for reference, already computed in notebook 03, not recomputed here):
  hotspot: rho=+0.0318  p=0.2837  n=1,136
  rare: rho=+0.2752  p=7.9e-17  n=884


## Figure 4: weighting robustness -- MCC across every weighted/unweighted comparison

The key new figure. Source: `results/main/{hotspot,rare}/metrics.json`,
`results/unweighted/{hotspot,rare}/metrics.json`,
`results/cpg/hotspot_unweighted_check_comparison.csv`,
`results/cpg/unweighted_check_comparison.csv` -- every MCC value plotted was
already computed and saved by an earlier notebook; this cell only reads and
plots them.


In [8]:
def load_json(*parts):
    with open(os.path.join(RESULTS_ROOT, *parts)) as f:
        return json.load(f)


unweighted_summary = pd.read_csv(os.path.join(RESULTS_ROOT, 'unweighted', 'summary.csv'))
rare_cpg_check = pd.read_csv(os.path.join(RESULTS_ROOT, 'cpg', 'unweighted_check_comparison.csv'))
hotspot_cpg_check = pd.read_csv(os.path.join(RESULTS_ROOT, 'cpg', 'hotspot_unweighted_check_comparison.csv'))

def get_check_mcc(df, subset_name, weighting_substr):
    row = df[(df['dataset_subset'] == subset_name) & (df['weighting'].str.contains(weighting_substr, regex=False))]
    return float(row['mcc'].iloc[0])

robustness_rows = [
    {
        'category': 'Main\n(hotspot)', 'dataset': 'hotspot',
        'weighted_mcc': load_json('main', 'hotspot', 'metrics.json')['mcc'],
        'unweighted_mcc': float(unweighted_summary.loc[
            (unweighted_summary['dataset'] == 'hotspot') & (unweighted_summary['weighting'] == 'unweighted'), 'mcc'
        ].iloc[0]),
    },
    {
        'category': 'Main\n(rare)', 'dataset': 'rare',
        'weighted_mcc': load_json('main', 'rare', 'metrics.json')['mcc'],
        'unweighted_mcc': float(unweighted_summary.loc[
            (unweighted_summary['dataset'] == 'rare') & (unweighted_summary['weighting'] == 'unweighted'), 'mcc'
        ].iloc[0]),
    },
    {
        'category': 'CpG\n(hotspot)', 'dataset': 'hotspot',
        'weighted_mcc': get_check_mcc(hotspot_cpg_check, 'hotspot-CpG', 'weighted ('),
        'unweighted_mcc': get_check_mcc(hotspot_cpg_check, 'hotspot-CpG', 'unweighted'),
    },
    {
        'category': 'CpG\n(rare)', 'dataset': 'rare',
        'weighted_mcc': get_check_mcc(rare_cpg_check, 'rare-CpG', 'weighted ('),
        'unweighted_mcc': get_check_mcc(rare_cpg_check, 'rare-CpG', 'unweighted'),
    },
]
robustness_df = pd.DataFrame(robustness_rows)
print(robustness_df)


          category  dataset  weighted_mcc  unweighted_mcc
0  Main\n(hotspot)  hotspot      0.066582       -0.055329
1     Main\n(rare)     rare      0.026101        0.140388
2   CpG\n(hotspot)  hotspot      0.089331        0.060996
3      CpG\n(rare)     rare      0.000000        0.000000


In [9]:
fig, ax = plt.subplots(figsize=(4.5, 3.2))

x_positions = np.arange(len(robustness_df))
ax.axhline(0, color=COLOR_REFERENCE, linewidth=0.8, linestyle='--', zorder=0)

for i, row in robustness_df.iterrows():
    color = DATASET_COLORS[row['dataset']]
    ax.plot([i, i], [row['weighted_mcc'], row['unweighted_mcc']], color=color, linewidth=1.2, zorder=1)
    ax.scatter([i], [row['weighted_mcc']], color=color, marker='o', s=45, zorder=2, label='weighted' if i == 0 else None)
    ax.scatter([i], [row['unweighted_mcc']], facecolors='none', edgecolors=color, marker='o', s=45,
               linewidths=1.4, zorder=2, label='unweighted' if i == 0 else None)

ax.set_xticks(x_positions)
ax.set_xticklabels(robustness_df['category'])
ax.set_ylabel('MCC')

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='black', markeredgecolor='black',
           markersize=6, label='weighted', linestyle='none'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='none', markeredgecolor='black',
           markersize=6, label='unweighted', linestyle='none'),
]
ax.legend(handles=legend_handles, frameon=False, loc='upper right')

save_fig(fig, 'fig_weighting_robustness')


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_weighting_robustness.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_weighting_robustness.svg


## Figure 5: CpG summary (weighted, descriptive)

Kept accuracy/majority-baseline focused and weighted-only -- the MCC
robustness story is already Figure 4's job, so this figure stays a clean,
non-duplicative descriptive view. Source: `results/cpg/metrics.csv` only.


In [10]:
cpg_metrics = pd.read_csv(os.path.join(RESULTS_ROOT, 'cpg', 'metrics.csv'))

fig, ax = plt.subplots(figsize=(4.2, 3.0))
datasets = ['hotspot', 'rare']
bar_width = 0.35
x_positions = np.arange(len(datasets))

for i, cpg_flag in enumerate((True, False)):
    heights = [cpg_metrics.loc[(cpg_metrics['dataset'] == name) & (cpg_metrics['is_cpg'] == cpg_flag), 'accuracy'].iloc[0]
               for name in datasets]
    baselines = [cpg_metrics.loc[(cpg_metrics['dataset'] == name) & (cpg_metrics['is_cpg'] == cpg_flag), 'majority_accuracy'].iloc[0]
                 for name in datasets]
    offset = (i - 0.5) * bar_width
    colors = [DATASET_COLORS[name] for name in datasets]
    alphas = 1.0 if cpg_flag else 0.45
    bars = ax.bar(x_positions + offset, heights, bar_width,
                   color=colors, alpha=(1.0 if cpg_flag else 0.45),
                   edgecolor='black', linewidth=0.4,
                   label='CpG' if cpg_flag else 'non-CpG')
    for xp, base in zip(x_positions + offset, baselines):
        ax.plot([xp - bar_width / 2, xp + bar_width / 2], [base, base], color=COLOR_REFERENCE, linewidth=1.4)

ax.plot([], [], color=COLOR_REFERENCE, linewidth=1.4, label='majority baseline (subset)')
ax.set_xticks(x_positions)
ax.set_xticklabels(datasets)
ax.set_ylabel('Accuracy')
ax.legend(frameon=False, fontsize=6.5)

save_fig(fig, 'fig_cpg_summary')


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_cpg_summary.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_cpg_summary.svg


## Figure 6: data pipeline flow (raw COSMIC rows -> per-window instance CSVs)

The paper's main methodological contribution figure: shows every filtering
stage notebook 00 applies, with the isoform-redundancy locus-clustering
collapse (13,423 positions -> 906 true loci) visually emphasized, not just
stated in text. Every number here is read from
`data/processed/pipeline_summary.json` and `data/processed/cluster_table.csv`
-- both already-saved outputs, no new computation.


In [11]:
pipeline_summary = json.load(open(os.path.join(PROJECT_ROOT, 'data', 'processed', 'pipeline_summary.json')))
cluster_table_counts = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'cluster_table.csv'))['hotspot_flag'].value_counts()

n_raw_rows = pipeline_summary['extraction_stats']['total_rows']
n_non_substitution = pipeline_summary['extraction_stats']['non_substitution']
n_valid_substitutions = pipeline_summary['extraction_stats']['valid_substitutions']
n_raw_positions = 14355  # printed by notebook 00 ("14,355 raw positions before N-filtering"); not stored as its own JSON field
n_valid_positions = pipeline_summary['n_positions_before_clustering']
n_dropped_by_nfilter = n_raw_positions - n_valid_positions
n_clusters = pipeline_summary['n_clusters_after_clustering']
n_hotspot_clusters = int(cluster_table_counts[True])
n_rare_clusters = int(cluster_table_counts[False])

# Final instance-row total (identical across all 4 window sizes, since the
# shared cluster-based split guarantees the same position set everywhere).
final_w21 = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'tp53_mutation_dataset_w21.csv'))
n_final_instances = len(final_w21)

print(f"raw rows: {n_raw_rows:,}")
print(f"valid substitutions: {n_valid_substitutions:,} (removed {n_non_substitution:,} non-substitution)")
print(f"raw positions: {n_raw_positions:,}")
print(f"valid positions (101bp N-filter): {n_valid_positions:,} (dropped {n_dropped_by_nfilter:,})")
print(f"true loci (11bp clustering): {n_clusters:,}")
print(f"hotspot / rare clusters: {n_hotspot_clusters:,} / {n_rare_clusters:,}")
print(f"final instance rows: {n_final_instances:,}")


raw rows: 88,218
valid substitutions: 30,650 (removed 57,568 non-substitution)
raw positions: 14,355
valid positions (101bp N-filter): 13,423 (dropped 932)
true loci (11bp clustering): 906
hotspot / rare clusters: 461 / 445
final instance rows: 693,428


In [12]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(7.8, 10.8))  # close to A4 portrait proportions (8.27 x 11.69in) with margin
ax.set_xlim(0, 10)
ax.set_ylim(0, 100)
ax.axis('off')

BOX_GRAY = '#e8e8e8'
BOX_EDGE = '#333333'
STAGE_W, STAGE_H = 8.6, 7.5


def draw_box(cx, cy, text, facecolor=BOX_GRAY, w=STAGE_W, h=STAGE_H, fontsize=7.3,
             fontweight='normal', textcolor='black'):
    box = FancyBboxPatch(
        (cx - w / 2, cy - h / 2), w, h,
        boxstyle='round,pad=0,rounding_size=0.5',
        facecolor=facecolor, edgecolor=BOX_EDGE, linewidth=0.8, zorder=2,
    )
    ax.add_patch(box)
    ax.text(cx, cy, text, ha='center', va='center', fontsize=fontsize,
            fontweight=fontweight, color=textcolor, zorder=3)


def draw_arrow(x, y_top, y_bottom, label=None, color='black', lw=0.9, label_fontsize=6.3, label_dx=0.15):
    arrow = FancyArrowPatch(
        (x, y_top), (x, y_bottom),
        arrowstyle='-|>', mutation_scale=10, color=color, linewidth=lw, zorder=1,
    )
    ax.add_patch(arrow)
    if label:
        ax.text(x + label_dx, (y_top + y_bottom) / 2, label, ha='left', va='center',
                fontsize=label_fontsize, color=color)


cx = 5.0

# Stage 1: raw rows
y1 = 95
draw_box(cx, y1, f"Raw COSMIC mutation rows\nn = {n_raw_rows:,}")

# Arrow 1->2: substitution filter
y2 = 81
draw_arrow(cx, y1 - STAGE_H / 2, y2 + STAGE_H / 2,
           label=f"-{n_non_substitution:,} indels /\ncomplex mutations")
draw_box(cx, y2, f"Substitution filter\nn = {n_valid_substitutions:,} valid substitutions")

# Arrow 2->3: position_id assignment
y3 = 67
draw_arrow(cx, y2 - STAGE_H / 2, y3 + STAGE_H / 2, label="position_id =\ngene_number_cdsPos")
draw_box(cx, y3, f"position_id assignment\n(transcript, CDS position)\nn = {n_raw_positions:,} raw positions")

# Arrow 3->4: 101bp N-filter
y4 = 53
draw_arrow(cx, y3 - STAGE_H / 2, y4 + STAGE_H / 2, label=f"-{n_dropped_by_nfilter:,} boundary N\n(101bp filter)")
draw_box(cx, y4, f"101bp N-filter\nn = {n_valid_positions:,} valid positions")

# Arrow 4->5: 11bp clustering -- THE key collapse, visually emphasized
y5 = 37
draw_arrow(cx, y4 - STAGE_H / 2, y5 + STAGE_H / 2 + 2.2,
           label=f"{n_valid_positions:,} -> {n_clusters:,}\n(isoform-redundancy fix)",
           color='#b2182b', lw=1.8, label_fontsize=7.2, label_dx=0.2)
draw_box(cx, y5, f"11bp exact-match locus clustering\nn = {n_clusters:,} true loci",
         facecolor='#f4c6c6', h=STAGE_H + 1.5, fontweight='bold')

# Arrow 5->6: hotspot/rare split (splits into two boxes)
y6 = 22
split_dx = 2.6
draw_arrow(cx - split_dx, y5 - (STAGE_H + 1.5) / 2, y6 + STAGE_H / 2)
draw_arrow(cx + split_dx, y5 - (STAGE_H + 1.5) / 2, y6 + STAGE_H / 2)
ax.text(cx, y5 - (STAGE_H + 1.5) / 2 - 1.4, f"hotspot/rare split (n_instances >= {pipeline_summary['hotspot_threshold']})",
        ha='center', va='top', fontsize=6.3)
draw_box(cx - split_dx, y6, f"Hotspot clusters\nn = {n_hotspot_clusters:,}", facecolor=COLOR_HOTSPOT, w=3.6, h=STAGE_H,
         fontsize=7.3, textcolor='white')
draw_box(cx + split_dx, y6, f"Rare clusters\nn = {n_rare_clusters:,}", facecolor=COLOR_RARE, w=3.6, h=STAGE_H,
         fontsize=7.3, textcolor='white')

# Arrow 6->7: expand to per-window instance CSVs
y7 = 8
draw_arrow(cx - split_dx, y6 - STAGE_H / 2, y7 + STAGE_H / 2)
draw_arrow(cx + split_dx, y6 - STAGE_H / 2, y7 + STAGE_H / 2)
ax.text(cx, y6 - STAGE_H / 2 - 1.4, "expand by window size (11/21/51/101bp) x COSMIC Count",
        ha='center', va='top', fontsize=6.3)
draw_box(cx, y7, f"Per-window instance CSVs\n(train/val/test x {{{'hotspot'}, {'rare'}}})\nn = {n_final_instances:,} instances",
         w=STAGE_W)

save_fig(fig, 'fig_pipeline_flow')


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_pipeline_flow.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_pipeline_flow.svg


## Confusion matrices (main experiment, hotspot and rare)

Reformatted, final versions of `results/main/{hotspot,rare}/confusion_matrix.png`
for the results-section writeup -- same underlying `confusion_matrix` array
from each dataset's `metrics.json` (no new computation, just row-normalizing
for color intensity, exactly as the original plot did), restyled to match
every other figure here: no baked-in title, print-journal font sizes, and
--- to tie into the paper's established hotspot=blue/rare=orange convention
-- each matrix uses a white-to-dataset-color colormap instead of a generic
blue colormap for both.


In [13]:
from matplotlib.colors import LinearSegmentedColormap

CMAP_HOTSPOT = LinearSegmentedColormap.from_list('white_to_hotspot', ['white', COLOR_HOTSPOT])
CMAP_RARE = LinearSegmentedColormap.from_list('white_to_rare', ['white', COLOR_RARE])
CMAPS = {'hotspot': CMAP_HOTSPOT, 'rare': CMAP_RARE}

for name in ['hotspot', 'rare']:
    metrics = load_json('main', name, 'metrics.json')
    cm = np.array(metrics['confusion_matrix'])
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    im = ax.imshow(cm_norm, cmap=CMAPS[name], vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASSES)))
    ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=45, ha='right')
    ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                     color='white' if cm_norm[i, j] > 0.5 else 'black', fontsize=6.0)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Row-normalized fraction', fontsize=7.5)
    cbar.ax.tick_params(labelsize=6.5)

    save_fig(fig, f'fig_confusion_matrix_{name}')


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_confusion_matrix_hotspot.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_confusion_matrix_hotspot.svg


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_confusion_matrix_rare.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_confusion_matrix_rare.svg


## Table: main results (`results/figures/table_main_results.csv`)

Reformatted from `results/main/summary.csv` for direct inclusion as a paper
table -- CI collapsed into one formatted string column.


In [14]:
main_summary_raw = pd.read_csv(os.path.join(RESULTS_ROOT, 'main', 'summary.csv'))

# Both CIs are shown: locus-(cluster-)resampled is primary (matches the text),
# instance-resampled is kept alongside to make the pseudoreplication effect
# on interval width visible, not to imply it's an equally valid estimate.
table_main = pd.DataFrame({
    'Dataset': main_summary_raw['dataset'],
    'Accuracy (95% CI, locus-resampled)': [
        f"{row.accuracy:.3f} ({row.accuracy_ci_low_clustered:.3f}-{row.accuracy_ci_high_clustered:.3f})"
        for row in main_summary_raw.itertuples()
    ],
    'Accuracy (95% CI, instance-resampled)': [
        f"{row.accuracy:.3f} ({row.accuracy_ci_low:.3f}-{row.accuracy_ci_high:.3f})"
        for row in main_summary_raw.itertuples()
    ],
    'Balanced accuracy': main_summary_raw['balanced_accuracy'].round(3),
    'Macro F1': main_summary_raw['macro_f1'].round(3),
    'Weighted F1': main_summary_raw['weighted_f1'].round(3),
    'MCC': main_summary_raw['mcc'].round(3),
    'Majority baseline': main_summary_raw['majority_accuracy'].round(3),
    'Random baseline': main_summary_raw['random_accuracy'].round(3),
    'CpG-rule baseline': main_summary_raw['cpg_accuracy'].round(3),
    'Logistic-regression baseline': main_summary_raw['logistic_accuracy'].round(3),
    'Position-majority ceiling': main_summary_raw['position_majority_ceiling'].round(3),
})

table_main_path = os.path.join(FIGURES_DIR, 'table_main_results.csv')
table_main.to_csv(table_main_path, index=False)
produced_files.append(table_main_path)
print(f"Saved -> {table_main_path}\n")
table_main


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\table_main_results.csv



,Dataset,"Accuracy (95% CI, locus-resampled)","Accuracy (95% CI, instance-resampled)",Balanced accuracy,Macro F1,Weighted F1,MCC,Majority baseline,Random baseline,CpG-rule baseline,Logistic-regression baseline,Position-majority ceiling
0,hotspot,0.132 (0.054-0.238),0.132 (0.130-0.134),0.260,0.155,0.094,0.067,0.649,0.166,0.649,0.193,0.803
1,rare,0.218 (0.152-0.292),0.218 (0.202-0.236),0.199,0.194,0.222,0.026,0.358,0.162,0.358,0.354,0.727


## Table: weighting robustness (`results/figures/table_weighting_robustness.csv`)


In [15]:
def interpret(weighted_mcc, unweighted_mcc, degenerate_note=None):
    collapsed = unweighted_mcc <= 0.05 or (weighted_mcc > 0 and unweighted_mcc <= 0) or \
                (abs(unweighted_mcc - weighted_mcc) > 0.15 and unweighted_mcc < weighted_mcc)
    if degenerate_note:
        return degenerate_note
    if weighted_mcc > 0.05 and unweighted_mcc > 0.05 and not collapsed:
        return 'robust -- conclusion unchanged'
    return 'did not survive -- treat as artifact'


table_robustness_rows = []
for _, row in robustness_df.iterrows():
    label = row['category'].replace('\n', ' ')
    degenerate_note = None
    if label == 'CpG (rare)':
        degenerate_note = 'did not survive -- degenerate (majority-reversion, n=24 too small)'
    table_robustness_rows.append({
        'Comparison': label,
        'Weighted MCC': round(row['weighted_mcc'], 4),
        'Unweighted MCC': round(row['unweighted_mcc'], 4),
        'Weighted accuracy': None,
        'Unweighted accuracy': None,
        'Interpretation': interpret(row['weighted_mcc'], row['unweighted_mcc'], degenerate_note),
    })

def get_check_value(df, subset_name, weighting_substr, column):
    row = df[(df['dataset_subset'] == subset_name) & (df['weighting'].str.contains(weighting_substr, regex=False))]
    return float(row[column].iloc[0])


# Fill in accuracy columns from the same already-saved sources (no new computation).
table_robustness_rows[0]['Weighted accuracy'] = round(load_json('main', 'hotspot', 'metrics.json')['accuracy'], 4)
table_robustness_rows[0]['Unweighted accuracy'] = round(float(unweighted_summary.loc[
    (unweighted_summary['dataset'] == 'hotspot') & (unweighted_summary['weighting'] == 'unweighted'), 'accuracy'].iloc[0]), 4)
table_robustness_rows[1]['Weighted accuracy'] = round(load_json('main', 'rare', 'metrics.json')['accuracy'], 4)
table_robustness_rows[1]['Unweighted accuracy'] = round(float(unweighted_summary.loc[
    (unweighted_summary['dataset'] == 'rare') & (unweighted_summary['weighting'] == 'unweighted'), 'accuracy'].iloc[0]), 4)
table_robustness_rows[2]['Weighted accuracy'] = round(get_check_value(hotspot_cpg_check, 'hotspot-CpG', 'weighted (', 'accuracy'), 4)
table_robustness_rows[2]['Unweighted accuracy'] = round(get_check_value(hotspot_cpg_check, 'hotspot-CpG', 'unweighted', 'accuracy'), 4)
table_robustness_rows[3]['Weighted accuracy'] = round(get_check_value(rare_cpg_check, 'rare-CpG', 'weighted (', 'accuracy'), 4)
table_robustness_rows[3]['Unweighted accuracy'] = round(get_check_value(rare_cpg_check, 'rare-CpG', 'unweighted', 'accuracy'), 4)

table_robustness = pd.DataFrame(table_robustness_rows)
table_robustness_path = os.path.join(FIGURES_DIR, 'table_weighting_robustness.csv')
table_robustness.to_csv(table_robustness_path, index=False)
produced_files.append(table_robustness_path)
print(f"Saved -> {table_robustness_path}\n")
table_robustness


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\table_weighting_robustness.csv



,Comparison,Weighted MCC,Unweighted MCC,Weighted accuracy,Unweighted accuracy,Interpretation
0,Main (hotspot),0.0666,-0.0553,0.1322,0.1602,did not survive -- treat as artifact
1,Main (rare),0.0261,0.1404,0.2180,0.3908,did not survive -- treat as artifact
2,CpG (hotspot),0.0893,0.0610,0.0801,0.1063,robust -- conclusion unchanged
3,CpG (rare),0.0000,0.0000,0.3846,0.3846,did not survive -- degenerate (majority-revers...


## Figure 7: repeated split robustness

Source: `results/repeated_splits/per_split_results.csv` (notebook 07 -- 5 new
split seeds + the original seed=42 run, 6 splits x 2 datasets, no new
computation here).

In [16]:
repeated_df = pd.read_csv(os.path.join(RESULTS_ROOT, 'repeated_splits', 'per_split_results.csv'))

# Single shared panel (matches fig_window_ablation's style), hotspot/rare overlaid
# by colour as elsewhere in this notebook. Points are deliberately NOT joined
# split-to-split: split seed has no inherent order (unlike window size in
# Figure 2, a real continuous axis) -- a connecting line would visually imply
# a trend across seeds that doesn't exist, since each split is an independent,
# exchangeable resampling rather than a point along a meaningful axis.
fig, ax = plt.subplots(figsize=(4.6, 3.2))

seeds = sorted(repeated_df['split_seed'].unique())
x_by_seed = {s: i for i, s in enumerate(seeds)}

for name in ['hotspot', 'rare']:
    sub = repeated_df[repeated_df['dataset'] == name].sort_values('split_seed')
    x = [x_by_seed[s] for s in sub['split_seed']]
    ax.scatter(x, sub['accuracy'], color=DATASET_COLORS[name], s=45, zorder=3, label=f'{name} CNN')
    ax.scatter(x, sub['majority_accuracy'], color=DATASET_COLORS[name], marker='_', s=160,
               linewidths=2, zorder=2, alpha=0.6, label=f'{name} majority baseline')

ax.set_xticks(range(len(seeds)))
ax.set_xticklabels([str(int(s)) for s in seeds])
ax.set_xlabel('Split seed')
ax.set_ylabel('Accuracy')
ax.legend(frameon=False, fontsize=6, ncol=1, loc='upper left', bbox_to_anchor=(1.02, 1.0))
fig.tight_layout()
save_fig(fig, 'fig_repeated_splits')


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_repeated_splits.png
Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_repeated_splits.svg


## Table 6: repeated split robustness (`results/figures/table_repeated_splits.csv`)

Per-split accuracy/MCC/majority-baseline/significance, plus a mean +/- SD
summary row per dataset. Source: `results/repeated_splits/per_split_results.csv`
and `aggregate_summary.csv` (notebook 07); no new computation here.

In [17]:
def fmt_range(mean, std, lo, hi, decimals=3):
    return f"{mean:.{decimals}f} +/- {std:.{decimals}f} (range {lo:.{decimals}f}-{hi:.{decimals}f})"

table_repeated_rows = []
for name in ['hotspot', 'rare']:
    sub = repeated_df[repeated_df['dataset'] == name]
    n_sig = int((sub['cluster_bootstrap_pvalue'] < 0.05).sum())
    n_beats = int(sub['cnn_beats_majority'].sum())
    table_repeated_rows.append({
        'Dataset': name,
        'n splits': len(sub),
        'Accuracy (mean +/- SD, range)': fmt_range(sub['accuracy'].mean(), sub['accuracy'].std(),
                                                     sub['accuracy'].min(), sub['accuracy'].max()),
        'MCC (mean +/- SD, range)': fmt_range(sub['mcc'].mean(), sub['mcc'].std(),
                                                sub['mcc'].min(), sub['mcc'].max()),
        'Majority baseline (mean +/- SD, range)': fmt_range(sub['majority_accuracy'].mean(), sub['majority_accuracy'].std(),
                                                               sub['majority_accuracy'].min(), sub['majority_accuracy'].max()),
        'Splits CNN beat majority': f"{n_beats}/{len(sub)}",
        'Splits significant (cluster-bootstrap p<0.05)': f"{n_sig}/{len(sub)}",
    })

table_repeated = pd.DataFrame(table_repeated_rows)
table_repeated_path = os.path.join(FIGURES_DIR, 'table_repeated_splits.csv')
table_repeated.to_csv(table_repeated_path, index=False)
produced_files.append(table_repeated_path)
print(f"Saved -> {table_repeated_path}\n")
print("Full per-split detail (not printed in the paper table) remains available at:")
print(f"  {os.path.join(RESULTS_ROOT, 'repeated_splits', 'per_split_results.csv')}")
print()
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
table_repeated


Saved -> C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\table_repeated_splits.csv

Full per-split detail (not printed in the paper table) remains available at:
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\repeated_splits\per_split_results.csv



,Dataset,n splits,"Accuracy (mean +/- SD, range)","MCC (mean +/- SD, range)","Majority baseline (mean +/- SD, range)",Splits CNN beat majority,Splits significant (cluster-bootstrap p<0.05)
0,hotspot,6,0.192 +/- 0.107 (range 0.090-0.361),0.037 +/- 0.095 (range -0.049-0.190),0.571 +/- 0.093 (range 0.454-0.685),0/6,4/6
1,rare,6,0.218 +/- 0.053 (range 0.148-0.289),0.021 +/- 0.058 (range -0.044-0.111),0.370 +/- 0.029 (range 0.351-0.428),0/6,4/6


## Final summary: every file produced, and any missing inputs

In [18]:
print("=" * 100)
print("FILES PRODUCED")
print("=" * 100)
for path in produced_files:
    print(f"  {path}")

print("\n" + "=" * 100)
print("MISSING INPUTS")
print("=" * 100)
if missing:
    print(f"  {len(missing)} input(s) were missing (see FileNotFoundError above -- execution would "
          "have stopped before this cell if that happened):")
    for m in missing:
        print(f"    {m}")
else:
    print("  None -- every required input listed at the top of this notebook was found on disk.")


FILES PRODUCED
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_class_distribution.png
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_class_distribution.svg
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_theoretical_ceiling.png
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_theoretical_ceiling.svg
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_window_ablation.png
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_window_ablation.svg
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_position_analysis.png
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_position_analysis.svg
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_weighting_robustness.png
  C:\Users\danya\Documents\projects\tp53_mutation_subtype\results\figures\fig_we